**Phase-01**

In [ ]:
# ==============================
# PHASE 0 / PHASE 01 - CHUNK 01
# Drive Mount & Environment Setup (FINAL)
# ==============================

import os
import torch
from google.colab import drive

# 1. MOUNT GOOGLE DRIVE (CODE-BASED, RESUME-SAFE)
drive.mount('/content/drive')

# 2. DEFINE ROOT PATHS (LOCKED)
PROJECT_ROOT = "/content/drive/MyDrive/Human-Fall-Recognition"
DATASET_ROOT = os.path.join(PROJECT_ROOT, "GUB-STFN-Fall-Dataset")

# 3. DEFINE ALL REQUIRED PROJECT PATHS (FINAL ROADMAP)
PATHS = {
    "Skeleton_Raw":  os.path.join(PROJECT_ROOT, "Skeleton_Raw"),
    "Sequences":     os.path.join(PROJECT_ROOT, "Sequences"),
    "Models":        os.path.join(PROJECT_ROOT, "Models"),
    "Output_Videos": os.path.join(PROJECT_ROOT, "Output_Videos"),
    "Logs":          os.path.join(PROJECT_ROOT, "Logs"),
    "Results":       os.path.join(PROJECT_ROOT, "Results"),
    "Figures":       os.path.join(PROJECT_ROOT, "Results", "Figures"),
    "Metrics":       os.path.join(PROJECT_ROOT, "Results", "Metrics")
}

# 4. CREATE ALL DIRECTORIES (SAFE IF THEY ALREADY EXIST)
print("\n--- Creating / Verifying Project Directories ---")
for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"✅ {name}")

# 5. DATASET SANITY CHECK
print("\n--- Dataset Check ---")
if not os.path.exists(DATASET_ROOT):
    raise FileNotFoundError(f"❌ Dataset folder not found: {DATASET_ROOT}")

for cls in ["NoFall", "Faint", "Slip", "Trip"]:
    cls_path = os.path.join(DATASET_ROOT, cls)
    if not os.path.exists(cls_path):
        raise FileNotFoundError(f"❌ Missing class folder: {cls}")
    print(f"✅ {cls}: {len(os.listdir(cls_path))} files")

# 6. GPU CHECK
print("\n--- Hardware Check ---")
if torch.cuda.is_available():
    print(f"✅ GPU Ready: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU not detected (OK for now, required later)")

print("\n==============================")
print("PHASE 0 SETUP COMPLETE")

In [ ]:
# ==============================
# PHASE 01 - CHUNK 02
# Dataset Validation & Metadata
# ==============================

import os
import cv2
import pandas as pd
from tqdm.notebook import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/Human-Fall-Recognition"
DATASET_ROOT = os.path.join(PROJECT_ROOT, "GUB-STFN-Fall-Dataset")
LOGS_DIR = os.path.join(PROJECT_ROOT, "Logs")
OUT_CSV = os.path.join(LOGS_DIR, "dataset_validation_log.csv")

CLASS_NAMES = ["NoFall", "Faint", "Slip", "Trip"]
VALID_EXTENSIONS = (".mp4", ".avi", ".mov", ".mkv")

records = []
print(f"Scanning dataset: {DATASET_ROOT}")

for cls in CLASS_NAMES:
    cls_path = os.path.join(DATASET_ROOT, cls)
    files = [f for f in sorted(os.listdir(cls_path)) if f.lower().endswith(VALID_EXTENSIONS)]

    for fname in tqdm(files, desc=cls):
        fpath = os.path.join(cls_path, fname)

        try:
            cap = cv2.VideoCapture(fpath)
            if not cap.isOpened():
                records.append([fname, cls, 0, 0, "BROKEN"])
                continue

            fps = cap.get(cv2.CAP_PROP_FPS)
            frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
            duration = frames / fps if fps > 0 else 0

            status = "OK" if duration > 0 else "BROKEN"
            records.append([fname, cls, round(duration, 2), round(fps, 2), status])
            cap.release()

        except Exception:
            records.append([fname, cls, 0, 0, "ERROR"])

df = pd.DataFrame(
    records,
    columns=["video_name", "class", "duration_sec", "fps", "status"]
)

df.to_csv(OUT_CSV, index=False)

print("Validation complete")
print(df["status"].value_counts())
print(f"Saved: {OUT_CSV}")

In [ ]:
# ==============================
# PHASE 01 - CHUNK 03
# Dataset Statistics & Figures (FINAL)
# ==============================

import os
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = "/content/drive/MyDrive/Human-Fall-Recognition"
LOGS_DIR = os.path.join(PROJECT_ROOT, "Logs")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "Results", "Figures")
METRICS_DIR = os.path.join(PROJECT_ROOT, "Results", "Metrics")

CSV_IN = os.path.join(LOGS_DIR, "dataset_validation_log.csv")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

# 1. LOAD VALID DATA
df = pd.read_csv(CSV_IN)
df = df[df["status"] == "OK"]

# 2. STATISTICS TABLE (FOR CHAPTER 3)
stats = df.groupby("class").agg(
    total_videos=("video_name", "count"),
    min_duration=("duration_sec", "min"),
    max_duration=("duration_sec", "max"),
    mean_duration=("duration_sec", "mean")
).reset_index()

stats["mean_duration"] = stats["mean_duration"].round(2)

stats_path = os.path.join(METRICS_DIR, "dataset_statistics.csv")
stats.to_csv(stats_path, index=False)

print("Statistics saved:", stats_path)
print(stats)

# 3. CLASS DISTRIBUTION FIGURE
plt.figure(figsize=(6,4))
counts = df["class"].value_counts().sort_index()
bars = plt.bar(counts.index, counts.values)

for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height(),
             int(bar.get_height()),
             ha='center', va='bottom')

plt.title("Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Videos")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "class_distribution.png"), dpi=300)
plt.close()

# 4. DURATION DISTRIBUTION FIGURE
plt.figure(figsize=(6,4))
plt.hist(df["duration_sec"], bins=30)
plt.title("Video Duration Distribution")
plt.xlabel("Duration (seconds)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "duration_distribution.png"), dpi=300)
plt.close()

print("Figures saved to:", FIGURES_DIR)